In [4]:
# Install the core Hugging Face libraries
#!pip install -q transformers[torch] datasets evaluate accelerate peft

import torch

# Check for Apple Silicon GPU (MPS)
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Apple Silicon GPU)")
else:
    device = torch.device("cpu")
    print("Using CPU")


Using MPS (Apple Silicon GPU)


In [ ]:
from transformers import pipeline

base_model_checkpoint = "distilbert-base-uncased"

base_classifier = pipeline(
    "sentiment-analysis",
    model=base_model_checkpoint,
    tokenizer=base_model_checkpoint,
    device=device
)

before_review = "This Browns quarterback can't throw passes to save his life"
base_output = base_classifier(before_review)[0]

print(f"Running on: {device}")
print("Tweet:", before_review)
print("Base model output:", base_output)

#Label 0 = Negative, Label 1 = Positive

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use mps


Running on: mps
Tweet: This Browns quarterback can't throw passes to save his life
Base model output: {'label': 'LABEL_0', 'score': 0.5377272963523865}


In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Load data
sentiment_df = pd.read_csv('/Users/alecxszhang/Desktop/Stat 359/student/Final_Project/nfl_sentiments.csv')

# 2. Select columns and drop rows with missing values
df = sentiment_df[['text', 'sentiment']].dropna()

# 3. Keep only the three main sentiment categories
# This removes the rare classes causing the "least populated class" error
valid_sentiments = ['neutral', 'negative', 'positive']
df = df[df['sentiment'].isin(valid_sentiments)]

# 4. First split: Separate the Test set (20% of total data)
# Stratify ensures the 20% test set has the same % of neg/neu/pos as the original
train_val, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['sentiment']
)

# 5. Second split: Separate the Remaining 80% into Train (60%) and Validation (20%)
# (0.25 of 80% = 20% of the total)
train_df, val_df = train_test_split(
    train_val, test_size=0.25, random_state=42, stratify=train_val['sentiment']
)

# Verify the results
print("Value counts after filtering:")
print(df['sentiment'].value_counts())
print("-" * 30)
print(f"Training set:   {len(train_df)} rows")
print(f"Validation set: {len(val_df)} rows")
print(f"Testing set:    {len(test_df)} rows")

Value counts after filtering:
sentiment
neutral     2235
negative    1757
positive    1161
Name: count, dtype: int64
------------------------------
Training set:   3091 rows
Validation set: 1031 rows
Testing set:    1031 rows


In [7]:
# Define label mapping
label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {v: k for k, v in label2id.items()}

# Add integer label column to each split
for split in [train_df, val_df, test_df]:
    split["label"] = split["sentiment"].map(label2id)

print("Label mapping:", label2id)
print(train_df[["text", "sentiment", "label"]].head())

Label mapping: {'negative': 0, 'neutral': 1, 'positive': 2}
                                                   text sentiment  label
4897  Fancy destroying the famous One Ring on your o...   neutral      1
1260  Police Union Urges Officers To Protest Miami D...   neutral      1
2047  Sean Payton compared Broncos wide receiver Dev...   neutral      1
722   #Jets fans may not be pleased with Aaron Rodge...  negative      0
2025  Whatever you need fundraising for GiveSendGo i...  positive      2


In [8]:
from transformers import AutoTokenizer
from datasets import Dataset  

tokenizer = AutoTokenizer.from_pretrained(base_model_checkpoint)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

# Convert pandas DataFrames → HuggingFace Datasets BEFORE calling .map()
tokenized_train_dataset = Dataset.from_pandas(train_df[["text", "label"]]).map(tokenize_function, batched=True)
tokenized_eval_dataset  = Dataset.from_pandas(val_df[["text", "label"]]).map(tokenize_function, batched=True)
tokenized_test_dataset  = Dataset.from_pandas(test_df[["text", "label"]]).map(tokenize_function, batched=True)

# Sanity check the tokenizer
sample_text = "The Browns finally won a game!"
sample_ids = tokenizer(sample_text, add_special_tokens=True)["input_ids"]
print("Sample text:", sample_text)
print("Encoded IDs:", sample_ids)
print("Decoded:", tokenizer.decode(sample_ids))
print("\nDataset features:", tokenized_train_dataset.features)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Map:   0%|          | 0/3091 [00:00<?, ? examples/s]

Map:   0%|          | 0/1031 [00:00<?, ? examples/s]

Map:   0%|          | 0/1031 [00:00<?, ? examples/s]

Sample text: The Browns finally won a game!
Encoded IDs: [101, 1996, 13240, 2633, 2180, 1037, 2208, 999, 102]
Decoded: [CLS] the browns finally won a game! [SEP]

Dataset features: {'text': Value(dtype='string', id=None), 'label': Value(dtype='int64', id=None), '__index_level_0__': Value(dtype='int64', id=None), 'input_ids': Sequence(feature=Value(dtype='int32', id=None), length=-1, id=None), 'attention_mask': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None)}


In [9]:
import numpy as np
import evaluate
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)

# ✅ Fix 1: Match the 3-class label mapping you defined in Cell 4
id2label = {0: "negative", 1: "neutral", 2: "positive"}
label2id = {"negative": 0, "neutral": 1, "positive": 2}

model = AutoModelForSequenceClassification.from_pretrained(
    base_model_checkpoint,
    num_labels=3,          # ✅ Fix 2: Was 2, must match actual number of classes
    id2label=id2label,
    label2id=label2id,
)
model.to(device)           # ✅ Fix 3: Explicitly move model to MPS/CUDA/CPU

# Technique 1: Freeze embeddings + lower transformer layers
num_frozen_layers = 4
for param in model.distilbert.embeddings.parameters():
    param.requires_grad = False
for layer in model.distilbert.transformer.layer[:num_frozen_layers]:
    for param in layer.parameters():
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Technique 2: Dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./distilbert-nfl-sentiment",   # ✅ Fix 4: Renamed to match your project
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    fp16=torch.cuda.is_available(),            # Safe: stays False on MPS/CPU
    lr_scheduler_type="cosine",
    warmup_steps=100,                          # ✅ Fix 5: Was 0.1 (float) — must be int
    max_grad_norm=1.0,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable params: 14,768,643 / 66,955,779 (22.06%)


In [10]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.640700,0.711113,0.687682
2,0.665000,0.642769,0.731329
3,0.590100,0.637510,0.741028


TrainOutput(global_step=582, training_loss=0.6962962554082838, metrics={'train_runtime': 166.8457, 'train_samples_per_second': 55.578, 'train_steps_per_second': 3.488, 'total_flos': 307098023493888.0, 'train_loss': 0.6962962554082838, 'epoch': 3.0})

In [22]:
from transformers import pipeline

# Step 1: Create the pipeline FIRST
fine_tuned_classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=device,
)

# Step 2: Run inference on the correct text
sample_text = "Mahomes just threw an interception"
after_output = fine_tuned_classifier(sample_text)[0]

print("Sample Text:", sample_text)
print("Before fine-tuning:", base_classifier(sample_text)[0])  
print("After fine-tuning:", after_output)                       

# Step 3: Evaluate and save
eval_results = trainer.evaluate()
print("Evaluation metrics:", eval_results)

trainer.save_model("./my_fine_tuned_model")
tokenizer.save_pretrained("./my_fine_tuned_model")

Device set to use mps


Sample Text: Mahomes just threw an interception
Before fine-tuning: {'label': 'LABEL_0', 'score': 0.5320447087287903}
After fine-tuning: {'label': 'negative', 'score': 0.6203907132148743}
Evaluation metrics: {'eval_loss': 0.6375102400779724, 'eval_accuracy': 0.7410281280310378, 'eval_runtime': 8.4432, 'eval_samples_per_second': 122.11, 'eval_steps_per_second': 7.698, 'epoch': 3.0}


('./my_fine_tuned_model/tokenizer_config.json',
 './my_fine_tuned_model/special_tokens_map.json',
 './my_fine_tuned_model/vocab.txt',
 './my_fine_tuned_model/added_tokens.json',
 './my_fine_tuned_model/tokenizer.json')

In [23]:
sample_text = "I hate this game so much"
after_output = fine_tuned_classifier(sample_text)[0]

print("Sample Text:", sample_text)
print("Before fine-tuning:", base_classifier(sample_text)[0])  
print("After fine-tuning:", after_output)        

Sample Text: I hate this game so much
Before fine-tuning: {'label': 'LABEL_0', 'score': 0.5538899302482605}
After fine-tuning: {'label': 'negative', 'score': 0.9421104788780212}
